# Plan Validation

Logical plan invariant checks used before execution.

In [ ]:
#| default_exp plan_validation

In [ ]:
#| export
from dataclasses import dataclass
from typing import List

from .lazy_expr import Expr
from .lazy_frame import (
    LogicalPlan,
    Scan,
    Filter,
    Project,
    Aggregate,
    Sort,
    Limit,
    Distinct,
    Join,
)


__all__ = ["PlanValidationError", "validate_plan", "validate_plan_or_raise"]


@dataclass
class PlanValidationError:
    """Represents a plan invariant violation."""

    path: str
    message: str


def validate_plan(plan: LogicalPlan) -> List[PlanValidationError]:
    """Validate LogicalPlan invariants and return all discovered errors."""
    errors: List[PlanValidationError] = []

    def walk(node: LogicalPlan, path: str) -> None:
        if isinstance(node, Scan):
            if node.table is None:
                errors.append(PlanValidationError(path, "Scan.table cannot be None"))
            return

        if isinstance(node, Filter):
            if node.predicate is None:
                errors.append(PlanValidationError(path, "Filter.predicate cannot be None"))
            elif not isinstance(node.predicate, Expr):
                errors.append(PlanValidationError(path, "Filter.predicate must be an Expr"))
            walk(node.input, f"{path}.input")
            return

        if isinstance(node, Project):
            if not node.exprs:
                errors.append(PlanValidationError(path, "Project.exprs cannot be empty"))
            if any(not isinstance(expr, Expr) for expr in node.exprs):
                errors.append(PlanValidationError(path, "Project.exprs must contain Expr nodes"))
            walk(node.input, f"{path}.input")
            return

        if isinstance(node, Aggregate):
            if not node.aggs:
                errors.append(PlanValidationError(path, "Aggregate.aggs cannot be empty"))
            if any(not isinstance(expr, Expr) for expr in node.group_by):
                errors.append(PlanValidationError(path, "Aggregate.group_by must contain Expr nodes"))
            if any(not isinstance(expr, Expr) for expr in node.aggs):
                errors.append(PlanValidationError(path, "Aggregate.aggs must contain Expr nodes"))
            walk(node.input, f"{path}.input")
            return

        if isinstance(node, Sort):
            if not node.by:
                errors.append(PlanValidationError(path, "Sort.by cannot be empty"))
            if len(node.descending) != len(node.by):
                errors.append(
                    PlanValidationError(
                        path,
                        "Sort.descending length must match Sort.by length",
                    )
                )
            if any(not isinstance(expr, Expr) for expr in node.by):
                errors.append(PlanValidationError(path, "Sort.by must contain Expr nodes"))
            walk(node.input, f"{path}.input")
            return

        if isinstance(node, Limit):
            if node.n < 0:
                errors.append(PlanValidationError(path, "Limit.n must be >= 0"))
            walk(node.input, f"{path}.input")
            return

        if isinstance(node, Distinct):
            if node.subset is not None and len(node.subset) != len(set(node.subset)):
                errors.append(PlanValidationError(path, "Distinct.subset contains duplicates"))
            walk(node.input, f"{path}.input")
            return

        if isinstance(node, Join):
            if node.how != "inner":
                errors.append(PlanValidationError(path, "Only inner joins are currently supported"))
            if not node.left_on or not node.right_on:
                errors.append(PlanValidationError(path, "Join key lists cannot be empty"))
            if len(node.left_on) != len(node.right_on):
                errors.append(
                    PlanValidationError(
                        path,
                        "Join.left_on and Join.right_on must have matching lengths",
                    )
                )
            walk(node.left, f"{path}.left")
            walk(node.right, f"{path}.right")
            return

        errors.append(PlanValidationError(path, f"Unsupported plan node: {type(node).__name__}"))

    walk(plan, "plan")
    return errors


def validate_plan_or_raise(plan: LogicalPlan) -> None:
    """Validate plan and raise ValueError with a compact error summary on failure."""
    errors = validate_plan(plan)
    if not errors:
        return

    details = "; ".join(f"{e.path}: {e.message}" for e in errors)
    raise ValueError(f"Invalid logical plan: {details}")